In [1]:
# 1. Uninstall everything related to PyG
!pip uninstall -y torch-geometric

# 2. Install ONLY the main library (modern PyG doesn't need the others for basic GCNs)
!pip install torch_geometric

# 3. NOW RESTART YOUR RUNTIME MANUALLY

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 63.7/63.7 kB 3.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 50.4 MB/s eta 0:00:00


In [2]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
import numpy as np
from tqdm import tqdm
from sklearn.metrics import average_precision_score
from torch_geometric.datasets import LRGBDataset
from torch_geometric.loader import DataLoader
from torch_geometric.utils import to_dense_batch, to_dense_adj
import os


In [3]:
import os

# ==========================================
# 1. CORE MODEL ARCHITECTURE (Graph-FNet)
# ==========================================

class SpectralMix(nn.Module):
    def __init__(self, dim):
        super().__init__()
        self.mlp = nn.Sequential(
            nn.Linear(dim, dim),
            nn.GELU(),
            nn.Linear(dim, dim)
        )
        self.norm = nn.LayerNorm(dim)

    def forward(self, x, U, mask=None):
        # x: [Batch, N, Dim]
        # U: [Batch, N, N]

        # 1. Project to Spectral Domain
        x_spec = torch.matmul(U.transpose(1, 2), x)

        # 2. Mix Frequency Features
        x_spec = self.mlp(x_spec)

        # 3. Inverse Transform
        x_out = torch.matmul(U, x_spec)

        # 4. Residual + Norm
        out = self.norm(x + x_out)

        if mask is not None:
            out = out * mask.unsqueeze(-1)
        return out

class DenseGCNLayer(nn.Module):
    def __init__(self, dim):
        super().__init__()
        self.lin = nn.Linear(dim, dim)

    def forward(self, x, A_norm):
        x = torch.matmul(A_norm, x)
        return F.relu(self.lin(x))

class HybridGraphFNet(nn.Module):
    def __init__(self, in_dim, hidden_dim=64, num_layers=3, out_dim=10):
        super().__init__()
        self.input_proj = nn.Linear(in_dim, hidden_dim)

        self.layers = nn.ModuleList([
            nn.ModuleDict({
                "local": DenseGCNLayer(hidden_dim),
                "global": SpectralMix(hidden_dim)
            })
            for _ in range(num_layers)
        ])

        self.alphas = nn.Parameter(torch.ones(num_layers, 1) * 0.5)
        self.classifier = nn.Linear(hidden_dim, out_dim)

    def compute_laplacian_basis(self, adj, mask):
        deg = adj.sum(dim=2)
        deg_inv_sqrt = torch.pow(deg + 1e-8, -0.5)
        deg_inv_sqrt[~mask] = 0
        D_inv_sqrt = torch.diag_embed(deg_inv_sqrt)

        A_norm = torch.matmul(torch.matmul(D_inv_sqrt, adj), D_inv_sqrt)

        I = torch.eye(adj.size(1), device=adj.device).unsqueeze(0)
        L = I - A_norm
        L = L * mask.unsqueeze(2) * mask.unsqueeze(1)

        # Eigendecomposition
        try:
            _, U = torch.linalg.eigh(L)
        except:
             # Fallback for very rare numerical instability
            U = torch.eye(adj.size(1), device=adj.device).unsqueeze(0).repeat(adj.size(0), 1, 1)

        return A_norm, U

    def forward(self, data):
        x, mask = to_dense_batch(data.x.float(), data.batch)
        adj = to_dense_adj(data.edge_index, data.batch, max_num_nodes=x.size(1))

        I = torch.eye(adj.size(1), device=x.device).unsqueeze(0)
        adj_loops = adj + I

        A_norm, U = self.compute_laplacian_basis(adj_loops, mask)

        x = self.input_proj(x)

        for idx, layer in enumerate(self.layers):
            x_local = layer["local"](x, A_norm)
            x_global = layer["global"](x, U, mask)

            alpha = torch.sigmoid(self.alphas[idx])
            x = alpha * x_local + (1 - alpha) * x_global

        x = x * mask.unsqueeze(-1)
        sum_pooled = x.sum(dim=1)
        num_nodes = mask.sum(dim=1, keepdim=True)
        graph_emb = sum_pooled / (num_nodes + 1e-6)

        return self.classifier(graph_emb)


# ==========================================
# 2. PEPTIDES-SPECIFIC ADAPTATIONS
# ==========================================

class SimpleAtomEncoder(nn.Module):
    def __init__(self, hidden_dim):
        super().__init__()
        self.embedding = nn.Embedding(512, hidden_dim)

    def forward(self, x):
        if x.dim() > 2:
            x = x[:, :, 0]
        return self.embedding(x.long())

class HybridGraphFNet_Peptides(HybridGraphFNet):
    def __init__(self, hidden_dim=64, num_layers=3, out_dim=10):
        super().__init__(in_dim=1, hidden_dim=hidden_dim, num_layers=num_layers, out_dim=out_dim)
        self.input_proj = SimpleAtomEncoder(hidden_dim)


# ==========================================
# 3. TRAINING SETUP (With TQDM)
# ==========================================

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

def load_data():
    print("Loading Peptides-func dataset...")
    # Using 'root' to store downloaded data
    train_dataset = LRGBDataset(root='./data/LRGB', name='Peptides-func', split='train')
    val_dataset = LRGBDataset(root='./data/LRGB', name='Peptides-func', split='val')
    test_dataset = LRGBDataset(root='./data/LRGB', name='Peptides-func', split='test')

    train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
    val_loader = DataLoader(val_dataset, batch_size=32)
    test_loader = DataLoader(test_dataset, batch_size=32)

    return train_loader, val_loader, test_loader

def evaluate(model, loader):
    model.eval()
    y_true = []
    y_pred = []

    with torch.no_grad():
        for batch in loader:
            batch = batch.to(device)
            out = model(batch)
            y_true.append(batch.y.cpu().numpy())
            y_pred.append(out.cpu().numpy())

    y_true = np.concatenate(y_true, axis=0)
    y_pred = np.concatenate(y_pred, axis=0)
    return average_precision_score(y_true, y_pred)

def train_peptides():
    train_loader, val_loader, test_loader = load_data()

    model = HybridGraphFNet_Peptides(hidden_dim=128, num_layers=4, out_dim=10).to(device)
    optimizer = optim.AdamW(model.parameters(), lr=0.001, weight_decay=1e-4)
    criterion = nn.BCEWithLogitsLoss()

    print(f"Starting Training on {device}...")

    best_val_ap = 0.0
    final_test_ap = 0.0

    epochs = 5

    for epoch in range(1, epochs + 1):
        model.train()
        total_loss = 0

        # --- TQDM WRAPPER HERE ---
        pbar = tqdm(train_loader, desc=f"Epoch {epoch}/{epochs}", unit="batch")

        for batch in pbar:
            batch = batch.to(device)
            optimizer.zero_grad()

            out = model(batch)
            loss = criterion(out, batch.y.float())

            loss.backward()
            optimizer.step()
            total_loss += loss.item()

            # Update progress bar with current loss
            pbar.set_postfix({"Loss": f"{loss.item():.4f}"})

        avg_loss = total_loss / len(train_loader)

        # Evaluate
        val_ap = evaluate(model, val_loader)
        test_ap = evaluate(model, test_loader)

        # Track Best
        if val_ap > best_val_ap:
            best_val_ap = val_ap
            final_test_ap = test_ap

        # Print summary after the progress bar closes
        print(f"Epoch {epoch:02d} Summary | Loss: {avg_loss:.4f} | Val AP: {val_ap:.4f} | Test AP: {test_ap:.4f}")

    print("-" * 50)
    print(f"Best Val AP: {best_val_ap:.4f}")
    print(f"Corresponding Test AP: {final_test_ap:.4f}")
    print(f"Hybrid Parameters={sum(p.numel() for p in model.parameters())}")

In [5]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [6]:

class SimpleAtomEncoder(nn.Module):
    def __init__(self, hidden_dim):
        super().__init__()
        self.embedding = nn.Embedding(512, hidden_dim)

    def forward(self, x):
        if x.dim() > 2:
            x = x[:, :, 0]
        return self.embedding(x.long())

In [7]:
def load_data():
    print("Loading Peptides-func dataset...")
    # Using 'root' to store downloaded data
    train_dataset = LRGBDataset(root='./data/LRGB', name='Peptides-func', split='train')
    val_dataset = LRGBDataset(root='./data/LRGB', name='Peptides-func', split='val')
    test_dataset = LRGBDataset(root='./data/LRGB', name='Peptides-func', split='test')

    train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
    val_loader = DataLoader(val_dataset, batch_size=32)
    test_loader = DataLoader(test_dataset, batch_size=32)

    return train_loader, val_loader, test_loader

def evaluate(model, loader):
    model.eval()
    y_true = []
    y_pred = []

    with torch.no_grad():
        for batch in loader:
            batch = batch.to(device)
            out = model(batch)
            y_true.append(batch.y.cpu().numpy())
            y_pred.append(out.cpu().numpy())

    y_true = np.concatenate(y_true, axis=0)
    y_pred = np.concatenate(y_pred, axis=0)
    return average_precision_score(y_true, y_pred)

In [8]:
class DenseGCNLayer(nn.Module):
    def __init__(self, dim):
        super().__init__()
        self.lin = nn.Linear(dim, dim)

    def forward(self, x, A_norm):
        # x: [B, N, D]
        # A_norm: [B, N, N]
        x = torch.matmul(A_norm, x)
        return F.relu(self.lin(x))


In [9]:
class SpectralMix(nn.Module):
    def __init__(self, dim):
        super().__init__()
        self.mlp = nn.Sequential(
            nn.Linear(dim, dim),
            nn.GELU(),
            nn.Linear(dim, dim)
        )
        self.norm = nn.LayerNorm(dim)

    def forward(self, x, U, mask=None):
        # x: [Batch, N, Dim]
        # U: [Batch, N, N]

        # 1. Project to Spectral Domain
        x_spec = torch.matmul(U.transpose(1, 2), x)

        # 2. Mix Frequency Features
        x_spec = self.mlp(x_spec)

        # 3. Inverse Transform
        x_out = torch.matmul(U, x_spec)

        # 4. Residual + Norm
        out = self.norm(x + x_out)

        if mask is not None:
            out = out * mask.unsqueeze(-1)
        return out

In [10]:
class LocalOnlyGCN_Batched(nn.Module):
    def __init__(self, hidden_dim=128, num_layers=4, out_dim=10):
        super().__init__()

        # Same atom encoder as Hybrid
        self.input_proj = SimpleAtomEncoder(hidden_dim)

        self.layers = nn.ModuleList([
            DenseGCNLayer(hidden_dim)
            for _ in range(num_layers)
        ])

        self.classifier = nn.Linear(hidden_dim, out_dim)

    def compute_normalized_adj(self, adj, mask):
        # adj: [B, N, N]
        deg = adj.sum(dim=2)
        deg_inv_sqrt = torch.pow(deg + 1e-8, -0.5)
        deg_inv_sqrt[~mask] = 0

        D_inv_sqrt = torch.diag_embed(deg_inv_sqrt)
        A_norm = torch.matmul(torch.matmul(D_inv_sqrt, adj), D_inv_sqrt)

        return A_norm

    def forward(self, data):
        # Dense batching
        x, mask = to_dense_batch(data.x, data.batch)
        adj = to_dense_adj(data.edge_index, data.batch, max_num_nodes=x.size(1))

        # Add self loops
        I = torch.eye(adj.size(1), device=adj.device).unsqueeze(0)
        adj = adj + I

        # Normalize adjacency
        A_norm = self.compute_normalized_adj(adj, mask)

        # Atom embedding
        x = self.input_proj(x)

        # Apply stacked local layers
        for layer in self.layers:
            x = layer(x, A_norm)

        # Mask padded nodes
        x = x * mask.unsqueeze(-1)

        # Global mean pooling
        sum_pooled = x.sum(dim=1)
        num_nodes = mask.sum(dim=1, keepdim=True)
        graph_emb = sum_pooled / (num_nodes + 1e-6)

        return self.classifier(graph_emb)


In [ ]:
def train_peptides():
    train_loader, val_loader, test_loader = load_data()

    model = LocalOnlyGCN_Batched(hidden_dim=200,num_layers=4,out_dim=10).to(device)
    optimizer = optim.AdamW(model.parameters(), lr=0.001, weight_decay=1e-4)
    criterion = nn.BCEWithLogitsLoss()

    print(f"Starting Training on {device}...")

    best_val_ap = 0.0
    final_test_ap = 0.0

    epochs = 5

    for epoch in range(1, epochs + 1):
        model.train()
        total_loss = 0

        # --- TQDM WRAPPER HERE ---
        pbar = tqdm(train_loader, desc=f"Epoch {epoch}/{epochs}", unit="batch")

        for batch in pbar:
            batch = batch.to(device)
            optimizer.zero_grad()

            out = model(batch)
            loss = criterion(out, batch.y.float())

            loss.backward()
            optimizer.step()
            total_loss += loss.item()

            # Update progress bar with current loss
            pbar.set_postfix({"Loss": f"{loss.item():.4f}"})

        avg_loss = total_loss / len(train_loader)

        # Evaluate
        val_ap = evaluate(model, val_loader)
        test_ap = evaluate(model, test_loader)

        # Track Best
        if val_ap > best_val_ap:
            best_val_ap = val_ap
            final_test_ap = test_ap

        # Print summary after the progress bar closes
        print(f"Epoch {epoch:02d} Summary | Loss: {avg_loss:.4f} | Val AP: {val_ap:.4f} | Test AP: {test_ap:.4f}")

    print("-" * 50)
    print(f"Best Val AP: {best_val_ap:.4f}")
    print(f"Corresponding Test AP: {final_test_ap:.4f}")
    print(f"Local only Parameters={sum(p.numel() for p in model.parameters())}")

In [ ]:
train_peptides()

Loading Peptides-func dataset...
Starting Training on cuda...


Epoch 1/5: 100%|██████████| 340/340 [00:06<00:00, 54.71batch/s, Loss=0.3378]


Epoch 01 Summary | Loss: 0.3652 | Val AP: 0.2026 | Test AP: 0.1988


Epoch 2/5: 100%|██████████| 340/340 [00:05<00:00, 59.21batch/s, Loss=0.3085]


Epoch 02 Summary | Loss: 0.3491 | Val AP: 0.2135 | Test AP: 0.2107


...

Epoch 03 Summary | Loss: 0.3442 | Val AP: 0.2862 | Test AP: 0.2810


Epoch 4/5: 100%|██████████| 340/340 [00:05<00:00, 66.34batch/s, Loss=0.3638]


Epoch 04 Summary | Loss: 0.3231 | Val AP: 0.3293 | Test AP: 0.3263


Epoch 5/5: 100%|██████████| 340/340 [00:05<00:00, 64.88batch/s, Loss=0.3169]


Epoch 05 Summary | Loss: 0.3117 | Val AP: 0.3516 | Test AP: 0.3532
--------------------------------------------------
Best Val AP: 0.3516
Corresponding Test AP: 0.3532
Local only Parameters=265210


In [11]:
class SpectralOnly_Peptides(nn.Module):
    def __init__(self, hidden_dim=128, num_layers=4, out_dim=10):
        super().__init__()

        self.input_proj = SimpleAtomEncoder(hidden_dim)

        self.layers = nn.ModuleList([
            SpectralMix(hidden_dim)
            for _ in range(num_layers)
        ])

        self.classifier = nn.Linear(hidden_dim, out_dim)

    def compute_laplacian_basis(self, adj, mask):
        deg = adj.sum(dim=2)
        deg_inv_sqrt = torch.pow(deg + 1e-8, -0.5)
        deg_inv_sqrt[~mask] = 0

        D_inv_sqrt = torch.diag_embed(deg_inv_sqrt)
        A_norm = torch.matmul(torch.matmul(D_inv_sqrt, adj), D_inv_sqrt)

        I = torch.eye(adj.size(1), device=adj.device).unsqueeze(0)
        L = I - A_norm

        L = L * mask.unsqueeze(2) * mask.unsqueeze(1)

        try:
            _, U = torch.linalg.eigh(L)
        except:
            U = torch.eye(adj.size(1), device=adj.device).unsqueeze(0).repeat(adj.size(0), 1, 1)

        return U

    def forward(self, data):
        x, mask = to_dense_batch(data.x, data.batch)
        adj = to_dense_adj(data.edge_index, data.batch, max_num_nodes=x.size(1))

        # Add self-loops
        I = torch.eye(adj.size(1), device=adj.device).unsqueeze(0)
        adj = adj + I

        # Compute eigenbasis
        U = self.compute_laplacian_basis(adj, mask)

        # Atom embedding
        x = self.input_proj(x)

        # Spectral-only stacking
        for layer in self.layers:
            x = layer(x, U, mask)

        # Mask padded nodes
        x = x * mask.unsqueeze(-1)

        # Global mean pooling
        sum_pooled = x.sum(dim=1)
        num_nodes = mask.sum(dim=1, keepdim=True)
        graph_emb = sum_pooled / (num_nodes + 1e-6)

        return self.classifier(graph_emb)

In [ ]:
def train_peptides():
    train_loader, val_loader, test_loader = load_data()

    model = SpectralOnly_Peptides(hidden_dim=128,num_layers=4,out_dim=10).to(device)

    optimizer = optim.AdamW(model.parameters(), lr=0.001, weight_decay=1e-4)
    criterion = nn.BCEWithLogitsLoss()

    print(f"Starting Training on {device}...")

    best_val_ap = 0.0
    final_test_ap = 0.0

    epochs = 5

    for epoch in range(1, epochs + 1):
        model.train()
        total_loss = 0

        # --- TQDM WRAPPER HERE ---
        pbar = tqdm(train_loader, desc=f"Epoch {epoch}/{epochs}", unit="batch")

        for batch in pbar:
            batch = batch.to(device)
            optimizer.zero_grad()

            out = model(batch)
            loss = criterion(out, batch.y.float())

            loss.backward()
            optimizer.step()
            total_loss += loss.item()

            # Update progress bar with current loss
            pbar.set_postfix({"Loss": f"{loss.item():.4f}"})

        avg_loss = total_loss / len(train_loader)

        # Evaluate
        val_ap = evaluate(model, val_loader)
        test_ap = evaluate(model, test_loader)

        # Track Best
        if val_ap > best_val_ap:
            best_val_ap = val_ap
            final_test_ap = test_ap

        # Print summary after the progress bar closes
        print(f"Epoch {epoch:02d} Summary | Loss: {avg_loss:.4f} | Val AP: {val_ap:.4f} | Test AP: {test_ap:.4f}")

    print("-" * 50)
    print(f"Best Val AP: {best_val_ap:.4f}")
    print(f"Corresponding Test AP: {final_test_ap:.4f}")
    print(f"Spectral only Parameters={sum(p.numel() for p in model.parameters())}")

In [ ]:
train_peptides()

Loading Peptides-func dataset...
Starting Training on cuda...


Epoch 1/5: 100%|██████████| 340/340 [01:03<00:00,  5.39batch/s, Loss=0.3540]


Epoch 01 Summary | Loss: 0.3564 | Val AP: 0.2978 | Test AP: 0.3001


Epoch 2/5: 100%|██████████| 340/340 [01:02<00:00,  5.46batch/s, Loss=0.2731]


Epoch 02 Summary | Loss: 0.3203 | Val AP: 0.3437 | Test AP: 0.3412


...

Epoch 03 Summary | Loss: 0.3029 | Val AP: 0.3665 | Test AP: 0.3664


Epoch 4/5: 100%|██████████| 340/340 [01:01<00:00,  5.54batch/s, Loss=0.3451]


Epoch 04 Summary | Loss: 0.2984 | Val AP: 0.3805 | Test AP: 0.3801


Epoch 5/5: 100%|██████████| 340/340 [01:01<00:00,  5.52batch/s, Loss=0.2628]


Epoch 05 Summary | Loss: 0.2931 | Val AP: 0.3846 | Test AP: 0.3722
--------------------------------------------------
Best Val AP: 0.3846
Corresponding Test AP: 0.3722
Spectral only Parameters=199946


In [12]:
from torch_geometric.datasets import LRGBDataset, ZINC

def load_dataset(dataset_name, batch_size=32):

    if dataset_name == "LRGB":
        train_dataset = LRGBDataset(root='./data/LRGB', name='Peptides-func', split='train')
        val_dataset   = LRGBDataset(root='./data/LRGB', name='Peptides-func', split='val')
        test_dataset  = LRGBDataset(root='./data/LRGB', name='Peptides-func', split='test')

        task_type = "multilabel"

    elif dataset_name == "ZINC":
        train_dataset = ZINC(root='./data/ZINC', split='train')
        val_dataset   = ZINC(root='./data/ZINC', split='val')
        test_dataset  = ZINC(root='./data/ZINC', split='test')

        task_type = "regression"

    else:
        raise ValueError("Unsupported dataset")

    train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
    val_loader   = DataLoader(val_dataset, batch_size=batch_size)
    test_loader  = DataLoader(test_dataset, batch_size=batch_size)

    return train_loader, val_loader, test_loader, task_type


In [13]:
def evaluate_multilabel(model, loader):
    model.eval()
    y_true, y_pred = [], []

    with torch.no_grad():
        for batch in loader:
            batch = batch.to(device)
            out = model(batch)
            y_true.append(batch.y.cpu().numpy())
            y_pred.append(out.cpu().numpy())

    y_true = np.concatenate(y_true, axis=0)
    y_pred = np.concatenate(y_pred, axis=0)

    return average_precision_score(y_true, y_pred)


def evaluate_regression(model, loader):
    model.eval()
    total_mae = 0
    total = 0

    with torch.no_grad():
        for batch in loader:
            batch = batch.to(device)
            out = model(batch)
            mae = F.l1_loss(out.squeeze(), batch.y.squeeze(), reduction="sum")
            total_mae += mae.item()
            total += batch.num_graphs

    return total_mae / total


In [14]:
import random

def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

def count_params(model):
    return sum(p.numel() for p in model.parameters())


In [15]:
def log_results_to_csv(filename, row_dict):

    file_exists = os.path.isfile(filename)

    with open(filename, mode='a', newline='') as file:
        writer = csv.DictWriter(file, fieldnames=row_dict.keys())

        if not file_exists:
            writer.writeheader()

        writer.writerow(row_dict)

In [16]:
import time

In [17]:
from tqdm import tqdm

def train_model(
    model_class,
    model_kwargs,
    dataset_name="LRGB",
    max_epochs=100,
    patience=10,
    batch_size=32,
    seeds=[2]
):

    train_loader, val_loader, test_loader, task_type = load_dataset(dataset_name, batch_size)

    results = []

    for seed in seeds:

        print("="*60)
        print(f"Dataset: {dataset_name} | Seed: {seed}")
        print("="*60)

        set_seed(seed)

        model = model_class(**model_kwargs).to(device)
        optimizer = optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-4)

        if task_type == "multilabel":
            criterion = nn.BCEWithLogitsLoss()
        else:
            criterion = nn.L1Loss()

        param_count = count_params(model)
        print(f"Parameters: {param_count}")

        best_val = float("inf") if task_type == "regression" else 0.0
        best_test = 0.0
        epochs_no_improve = 0
        best_epoch = 0

        torch.cuda.reset_peak_memory_stats()
        start_time = time.time()

        for epoch in range(1, max_epochs + 1):

            model.train()
            total_loss = 0

            epoch_start = time.time()

            pbar = tqdm(train_loader, desc=f"Seed {seed} | Epoch {epoch}", leave=False)

            for batch in pbar:
                batch = batch.to(device)
                optimizer.zero_grad()

                out = model(batch)

                if task_type == "multilabel":
                    loss = criterion(out, batch.y.float())
                else:
                    loss = criterion(out.squeeze(), batch.y.squeeze())

                loss.backward()
                optimizer.step()

                total_loss += loss.item()

                pbar.set_postfix({"Loss": f"{loss.item():.4f}"})

            avg_loss = total_loss / len(train_loader)

            # ----- Evaluation -----
            if task_type == "multilabel":
                val_metric = evaluate_multilabel(model, val_loader)
                test_metric = evaluate_multilabel(model, test_loader)
                improved = val_metric > best_val
            else:
                val_metric = evaluate_regression(model, val_loader)
                test_metric = evaluate_regression(model, test_loader)
                improved = val_metric < best_val

            epoch_time = time.time() - epoch_start

            print(f"Epoch {epoch:03d} | "
                  f"Loss {avg_loss:.4f} | "
                  f"Val {val_metric:.4f} | "
                  f"Test {test_metric:.4f} | "
                  f"Time {epoch_time:.2f}s")

            # ----- Early Stopping -----
            if improved:
                best_val = val_metric
                best_test = test_metric
                best_epoch = epoch
                epochs_no_improve = 0
            else:
                epochs_no_improve += 1

            if epochs_no_improve >= patience:
                print("Early stopping triggered.")
                break

        total_time = time.time() - start_time
        peak_mem = torch.cuda.max_memory_allocated() / 1024**2

        print("-"*60)
        print(f"Seed {seed} Results:")
        print(f"Best Val: {best_val:.4f}")
        print(f"Test @ Best Val: {best_test:.4f}")
        print(f"Best Epoch: {best_epoch}")
        print(f"Total Time: {total_time:.2f}s")
        print(f"Peak Memory: {peak_mem:.2f} MB")
        print("-"*60)

        results.append(best_test)

        # ----- CSV LOGGING (PER SEED) -----
        log_results_to_csv(
            "results.csv",
            {
                "Dataset": dataset_name,
                "Model": model_class.__name__,
                "HiddenDim": model_kwargs.get("hidden_dim"),
                "NumLayers": model_kwargs.get("num_layers"),
                "Params": param_count,
                "Seed": seed,
                "BestVal": best_val,
                "BestTest": best_test,
                "BestEpoch": best_epoch,
                "TotalTime": total_time,
                "PeakMemoryMB": peak_mem
            }
        )

    mean = np.mean(results)
    std = np.std(results)

    print("="*60)
    print(f"FINAL Test Metric: {mean:.4f} ± {std:.4f}")
    print("="*60)

    return mean, std

In [ ]:
train_model(
    HybridGraphFNet_Peptides,
    dict(hidden_dim=128, num_layers=4, out_dim=10),
    dataset_name="LRGB",
    max_epochs=100,
    patience=10
)


Processing...
Processing test dataset: 100%|██████████| 2331/2331 [00:00<00:00, 30618.33it/s]
Done!


Dataset: LRGB | Seed: 0
Parameters: 265998


Epoch 001 | Loss 0.3510 | Val 0.2906 | Test 0.2942 | Time 91.19s


Epoch 002 | Loss 0.3087 | Val 0.3621 | Test 0.3656 | Time 88.68s


Epoch 003 | Loss 0.2945 | Val 0.3794 | Test 0.3754 | Time 89.29s


...

Epoch 044 | Loss 0.1573 | Val 0.5257 | Test 0.5236 | Time 88.92s


Epoch 045 | Loss 0.1513 | Val 0.5095 | Test 0.5080 | Time 89.08s


Epoch 046 | Loss 0.1525 | Val 0.5095 | Test 0.5177 | Time 89.00s
Early stopping triggered.
------------------------------------------------------------
Seed 0 Results:
Best Val: 0.5320
Test @ Best Val: 0.5215
Best Epoch: 36
Total Time: 4103.75s
Peak Memory: 341.67 MB
------------------------------------------------------------
Dataset: LRGB | Seed: 1
Parameters: 265998


Epoch 001 | Loss 0.3528 | Val 0.2983 | Test 0.3046 | Time 89.26s


Seed 1 | Epoch 2:  45%|████▌     | 153/340 [00:29<00:38,  4.88it/s, Loss=0.3442]

In [ ]:
train_model(
    LocalOnlyGCN_Batched,
    dict(hidden_dim=200, num_layers=4, out_dim=10),
    dataset_name="LRGB",
    max_epochs=100,
    patience=10
)


In [ ]:
out_dim=1


In [ ]:
train_model(
    HybridGraphFNet,
    dict(in_dim=28, hidden_dim=128, num_layers=4, out_dim=1),
    dataset_name="ZINC",
    max_epochs=50,
    patience=10
)


In [ ]:
class SpectralOnly(nn.Module):
    def __init__(self, in_dim, hidden_dim=128, num_layers=4, out_dim=1, use_atom_encoder=False):
        super().__init__()

        self.use_atom_encoder = use_atom_encoder

        if use_atom_encoder:
            self.input_proj = SimpleAtomEncoder(hidden_dim)
        else:
            self.input_proj = nn.Linear(in_dim, hidden_dim)

        self.layers = nn.ModuleList([
            SpectralMix(hidden_dim)
            for _ in range(num_layers)
        ])

        self.classifier = nn.Linear(hidden_dim, out_dim)

    def compute_laplacian_basis(self, adj, mask):
        deg = adj.sum(dim=2)
        deg_inv_sqrt = torch.pow(deg + 1e-8, -0.5)
        deg_inv_sqrt[~mask] = 0

        D_inv_sqrt = torch.diag_embed(deg_inv_sqrt)
        A_norm = torch.matmul(torch.matmul(D_inv_sqrt, adj), D_inv_sqrt)

        I = torch.eye(adj.size(1), device=adj.device).unsqueeze(0)
        L = I - A_norm

        L = L * mask.unsqueeze(2) * mask.unsqueeze(1)

        try:
            _, U = torch.linalg.eigh(L)
        except:
            U = torch.eye(adj.size(1), device=adj.device).unsqueeze(0).repeat(adj.size(0), 1, 1)

        return U

    def forward(self, data):
        x, mask = to_dense_batch(data.x, data.batch)
        adj = to_dense_adj(data.edge_index, data.batch, max_num_nodes=x.size(1))

        # Add self-loops
        I = torch.eye(adj.size(1), device=adj.device).unsqueeze(0)
        adj = adj + I

        U = self.compute_laplacian_basis(adj, mask)

        x = self.input_proj(x)

        for layer in self.layers:
            x = layer(x, U, mask)

        x = x * mask.unsqueeze(-1)

        sum_pooled = x.sum(dim=1)
        num_nodes = mask.sum(dim=1, keepdim=True)
        graph_emb = sum_pooled / (num_nodes + 1e-6)

        return self.classifier(graph_emb)


In [ ]:
train_model(
    SpectralOnly,
    dict(
        in_dim=1,
        hidden_dim=128,
        num_layers=4,
        out_dim=10,
        use_atom_encoder=True
    ),
    dataset_name="LRGB",
    max_epochs=100,
    patience=10
)


Dataset: LRGB | Seed: 2
Parameters: 199946


Epoch 001 | Loss 0.3547 | Val 0.2928 | Test 0.2972 | Time 88.65s


Epoch 002 | Loss 0.3193 | Val 0.3191 | Test 0.3223 | Time 86.71s


Epoch 003 | Loss 0.3054 | Val 0.3660 | Test 0.3572 | Time 87.13s


Epoch 004 | Loss 0.2997 | Val 0.3754 | Test 0.3733 | Time 87.15s


Seed 2 | Epoch 5:  49%|████▊     | 165/340 [00:30<00:31,  5.56it/s, Loss=0.4295]

In [ ]:
train_model(
    SpectralOnly,
    dict(
        in_dim=28,
        hidden_dim=128,
        num_layers=4,
        out_dim=1,
        use_atom_encoder=False
    ),
    dataset_name="ZINC",
    max_epochs=50,
    patience=10
)


In [1]:
import csv
import os


In [3]:
import csv
import os
import numpy as np
rows = [
    {
        "Dataset": "LRGB", "Model": "HybridGraphFNet_Best_Peptides",
        "HiddenDim": 128, "NumLayers": 4, "Params": 328603,
        "Seed": 0, "BestVal": 0.6385, "TestAP": 0.6270,
        "BestEpoch": 66, "TotalTime": 6208.38, "PeakMemoryMB": 221.63,
        "GateCollapsed": False, "MaxEpochs": 150, "Patience": 20,
        "BatchSize": 8, "AccumSteps": 4, "EffectiveBatch": 32,
    },
    {
        "Dataset": "LRGB", "Model": "HybridGraphFNet_Best_Peptides",
        "HiddenDim": 128, "NumLayers": 4, "Params": 328603,
        "Seed": 1, "BestVal": 0.6430, "TestAP": 0.6317,
        "BestEpoch": 70, "TotalTime": 6508.60, "PeakMemoryMB": 225.69,
        "GateCollapsed": False, "MaxEpochs": 150, "Patience": 20,
        "BatchSize": 8, "AccumSteps": 4, "EffectiveBatch": 32,
    },
    {
        "Dataset": "LRGB", "Model": "HybridGraphFNet_Best_Peptides",
        "HiddenDim": 128, "NumLayers": 4, "Params": 328603,
        "Seed": 2, "BestVal": 0.6348, "TestAP": 0.6146,
        "BestEpoch": 95, "TotalTime": 8311.32, "PeakMemoryMB": 225.67,
        "GateCollapsed": False, "MaxEpochs": 150, "Patience": 20,
        "BatchSize": 8, "AccumSteps": 4, "EffectiveBatch": 32,
    },
]

filepath = "results.csv"
fieldnames = list(rows[0].keys())
file_exists = os.path.isfile(filepath)

with open(filepath, "a", newline="") as f:
    writer = csv.DictWriter(f, fieldnames=fieldnames)
    if not file_exists:
        writer.writeheader()
    writer.writerows(rows)

print("Logged 3 seeds to results.csv")
print(f"Test AP: {np.mean([r['TestAP'] for r in rows]):.4f} +/- {np.std([r['TestAP'] for r in rows]):.4f}")

Logged 3 seeds to results.csv
Test AP: 0.6244 +/- 0.0072


NEW COMPLETE CODE HOPEFULLY IT WORKSSS

In [19]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
import numpy as np
import time
from torch_geometric.utils import to_dense_batch, to_dense_adj
from torch_geometric.nn import GCNConv

In [20]:
class SimpleAtomEncoder(nn.Module):
    def __init__(self, hidden_dim):
        super().__init__()
        # Peptides-func atom features: 9 categorical features
        self.embeddings = nn.ModuleList([
            nn.Embedding(64, hidden_dim),   # atomic num
            nn.Embedding(10, hidden_dim),   # chirality
            nn.Embedding(10, hidden_dim),   # degree
            nn.Embedding(10, hidden_dim),   # formal charge
            nn.Embedding(10, hidden_dim),   # num Hs
            nn.Embedding(10, hidden_dim),   # num radical e
            nn.Embedding(10, hidden_dim),   # hybridization
            nn.Embedding(3,  hidden_dim),   # aromaticity
            nn.Embedding(10, hidden_dim),   # ring membership
        ])
        self.proj = nn.Linear(hidden_dim, hidden_dim)

    def forward(self, x):
        # x: [B, N, 9] integer features
        x = x.long().clamp(min=0)
        out = sum(emb(x[..., i]) for i, emb in enumerate(self.embeddings))
        return self.proj(F.gelu(out))

In [29]:
class DenseGCNLayer(nn.Module):
    def __init__(self, hidden_dim, edge_dim=3):
        super().__init__()
        self.node_lin = nn.Linear(hidden_dim, hidden_dim)
        # Project edge features to scalar weight [B,N,N] not [B,N,N,H]
        # Old approach created [B,N,N,H] intermediate = 2.6GB at batch=32
        self.edge_lin = nn.Linear(edge_dim, 1)
        self.norm     = nn.LayerNorm(hidden_dim)

    def forward(self, x, A_norm, edge_attr_dense=None):
        # x:               [B, N, H]
        # A_norm:          [B, N, N]
        # edge_attr_dense: [B, N, N, edge_dim]
        if edge_attr_dense is not None:
            # Scalar edge gate [B, N, N] — same memory as adj, not H times more
            E   = torch.sigmoid(self.edge_lin(edge_attr_dense)).squeeze(-1)  # [B,N,N]
            msg = torch.bmm(A_norm * E, x)                                   # [B,N,H]
        else:
            msg = torch.bmm(A_norm, x)

        return self.norm(F.gelu(self.node_lin(msg)))

In [22]:
class SpectralMixMH(nn.Module):
    def __init__(self, hidden_dim, num_heads=4):
        super().__init__()
        self.num_heads   = num_heads
        self.head_dim    = hidden_dim // num_heads
        self.filter_gen  = nn.Linear(hidden_dim, hidden_dim)
        self.out_proj    = nn.Linear(hidden_dim, hidden_dim)
        self.norm        = nn.LayerNorm(hidden_dim)

    def forward(self, x, U, mask):
        # x: [B, N, H],  U: [B, N, N]
        B, N, H = x.shape

        # Spectral domain: x_hat = U^T x
        x_hat     = torch.bmm(U.transpose(1, 2), x)         # [B, N, H]

        # Learned per-node spectral filter
        fil       = torch.sigmoid(self.filter_gen(x_hat))   # [B, N, H]
        x_filtered = fil * x_hat                             # [B, N, H]

        # Back to spatial: x_out = U x_filtered
        x_out = torch.bmm(U, x_filtered)                    # [B, N, H]

        # Zero out padded positions
        x_out = x_out * mask.unsqueeze(-1)

        return self.norm(self.out_proj(F.gelu(x_out)))

In [23]:
class AttentionPooling(nn.Module):
    def __init__(self, hidden_dim):
        super().__init__()
        self.gate = nn.Sequential(
            nn.Linear(hidden_dim, hidden_dim // 2),
            nn.Tanh(),
            nn.Linear(hidden_dim // 2, 1)
        )

    def forward(self, x, mask):
        # x:    [B, N, H]
        # mask: [B, N]  bool
        scores  = self.gate(x).squeeze(-1)                   # [B, N]
        scores  = scores.masked_fill(~mask, -1e9)
        weights = torch.softmax(scores, dim=1).unsqueeze(-1) # [B, N, 1]
        return (x * weights).sum(dim=1)                      # [B, H]

In [30]:
class HybridGraphFNet_Best(nn.Module):
    def __init__(
        self,
        hidden_dim  = 128,
        num_layers  = 4,
        out_dim     = 10,
        num_heads   = 4,
        lap_k       = 8,
        dropout     = 0.1,
        edge_dim    = 3,
    ):
        super().__init__()
        self.lap_k   = lap_k
        self.dropout = nn.Dropout(dropout)

        # --- Encoders ---
        self.input_proj  = SimpleAtomEncoder(hidden_dim)
        self.pe_encoder  = nn.Linear(lap_k, hidden_dim)

        # --- Layers ---
        self.layers = nn.ModuleList([
            nn.ModuleDict({
                "local":  DenseGCNLayer(hidden_dim, edge_dim=edge_dim),
                "global": SpectralMixMH(hidden_dim, num_heads=num_heads),
                "gate":   nn.Linear(hidden_dim, hidden_dim),
                "norm":   nn.LayerNorm(hidden_dim),
            })
            for _ in range(num_layers)
        ])

        # --- Readout ---
        self.pool = AttentionPooling(hidden_dim)
        self.classifier = nn.Sequential(
            nn.Linear(hidden_dim, hidden_dim),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim, out_dim)
        )

    # ------------------------------------------------------------------
    def compute_laplacian_basis(self, adj, mask):
      B, N, _ = adj.shape
      A_list, U_list = [], []

      for b in range(B):
          n = int(mask[b].sum().item())
          adj_b = adj[b, :n, :n]

          deg          = adj_b.sum(dim=1)
          deg_inv_sqrt = torch.pow(deg + 1e-8, -0.5)
          D_inv_sqrt   = torch.diag(deg_inv_sqrt)
          A_norm_b     = D_inv_sqrt @ adj_b @ D_inv_sqrt

          L_b = torch.eye(n, device=adj.device) - A_norm_b

          try:
              _, U_b = torch.linalg.eigh(L_b)

              # ---- Sign fix ----
              # For each eigenvector column, find the element with largest
              # absolute value and force it to be positive
              max_abs_idx = torch.abs(U_b).argmax(dim=0)          # [n]
              signs = torch.sign(
                  U_b[max_abs_idx, torch.arange(U_b.size(1), device=adj.device)]
              )                                                     # [n]
              signs[signs == 0] = 1.0                              # avoid multiply by 0
              U_b = U_b * signs.unsqueeze(0)                       # [n, n]

          except Exception:
              U_b = torch.eye(n, device=adj.device)

          # Pad back to N
          A_pad = F.pad(A_norm_b, (0, N - n, 0, N - n))
          U_pad = F.pad(U_b,      (0, N - n, 0, N - n))
          A_list.append(A_pad)
          U_list.append(U_pad)

      return torch.stack(A_list), torch.stack(U_list)

    # ------------------------------------------------------------------
    def forward(self, data):
        # ---- Dense conversion ----
        x,    mask = to_dense_batch(data.x.float(), data.batch)  # [B,N,F], [B,N]
        adj         = to_dense_adj(
            data.edge_index, data.batch,
            max_num_nodes=x.size(1)
        )                                                         # [B,N,N]

        # Edge attributes (3 features: bond type, stereo, is_aromatic)
        if data.edge_attr is not None:
            edge_attr_dense = to_dense_adj(
                data.edge_index, data.batch,
                edge_attr=data.edge_attr[:, :3].float(),
                max_num_nodes=x.size(1)
            )                                                     # [B,N,N,3]
        else:
            edge_attr_dense = None

        # Self-loops
        I   = torch.eye(adj.size(1), device=x.device).unsqueeze(0)
        adj = adj + I

        # ---- Spectral basis (clean, per-graph) ----
        A_norm, U = self.compute_laplacian_basis(adj, mask)

        # ---- Node features ----
        x = self.input_proj(x)                                   # [B,N,H]

        # ---- Laplacian PE injection ----
        k       = min(self.lap_k, U.size(-1))
        lap_pe  = U[:, :, :k]                                    # [B,N,k]
        lap_pe  = lap_pe * mask.unsqueeze(-1)
        x       = x + self.pe_encoder(lap_pe)                    # [B,N,H]

        # ---- Message passing ----
        for layer in self.layers:
            x_res    = x
            x_local  = layer["local"](x, A_norm, edge_attr_dense)
            x_global = layer["global"](x, U, mask)

            gate  = torch.sigmoid(layer["gate"](x))
            x_mix = gate * x_local + (1 - gate) * x_global

            x = layer["norm"](x_res + self.dropout(x_mix))

        # ---- Readout ----
        x = x * mask.unsqueeze(-1)
        graph_emb = self.pool(x, mask)                           # [B,H]

        return self.classifier(graph_emb)

In [26]:
class HybridGraphFNet_Best_Peptides(HybridGraphFNet_Best):
    def __init__(self, hidden_dim=128, num_layers=4, out_dim=10):
        super().__init__(
            hidden_dim = hidden_dim,
            num_layers = num_layers,
            out_dim    = out_dim,
            num_heads  = 4,
            lap_k      = 8,
            dropout    = 0.1,
            edge_dim   = 3,
        )
        # input_proj already set to SimpleAtomEncoder in parent

In [31]:
from tqdm import tqdm

def check_gate_health(model, val_loader, device):
    model.eval()
    batch = next(iter(val_loader)).to(device)

    with torch.no_grad():
        x, mask = to_dense_batch(batch.x.float(), batch.batch)
        adj = to_dense_adj(batch.edge_index, batch.batch, max_num_nodes=x.size(1))
        I   = torch.eye(adj.size(1), device=x.device).unsqueeze(0)
        adj = adj + I

        A_norm, U = model.compute_laplacian_basis(adj, mask)
        x_enc     = model.input_proj(x)
        k         = min(model.lap_k, U.size(-1))
        lap_pe    = U[:, :, :k] * mask.unsqueeze(-1)
        x_enc     = x_enc + model.pe_encoder(lap_pe)

        print("\n--- Gate Health Check ---")
        collapsed = False
        for i, layer in enumerate(model.layers):
            gate_vals = torch.sigmoid(layer["gate"](x_enc))
            mean_g    = gate_vals.mean().item()
            std_g     = gate_vals.std().item()

            if mean_g > 0.85:
                status    = "⚠️  COLLAPSED → GCN (spectral dead)"
                collapsed = True
            elif mean_g < 0.15:
                status    = "⚠️  COLLAPSED → SPECTRAL (GCN dead)"
                collapsed = True
            elif std_g < 0.05:
                status    = "⚠️  UNIFORM (not learning per-node routing)"
                collapsed = True
            else:
                status = "✅ HEALTHY"

            print(f"  Layer {i} | mean={mean_g:.4f} | std={std_g:.4f} | {status}")

        if collapsed:
            print("  ACTION: lr will be reset to 5e-4.")
        else:
            print("  All gates healthy.")
        print("-------------------------\n")

    model.train()
    return collapsed


def train_model(
    model_class,
    model_kwargs,
    dataset_name = "LRGB",
    max_epochs   = 120,
    patience     = 20,
    batch_size   = 8,
    accum_steps  = 4,
    seeds        = [0, 1, 2]
):
    train_loader, val_loader, test_loader, task_type = load_dataset(dataset_name, batch_size)

    results = []

    for seed in seeds:

        print("=" * 60)
        print(f"Dataset: {dataset_name} | Seed: {seed}")
        print(f"Batch size: {batch_size} | Accum steps: {accum_steps} | Effective batch: {batch_size * accum_steps}")
        print("=" * 60)

        set_seed(seed)

        model      = model_class(**model_kwargs).to(device)
        optimizer  = optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-4)
        scheduler  = optim.lr_scheduler.CosineAnnealingLR(
            optimizer, T_max=max_epochs, eta_min=1e-5
        )
        criterion  = nn.BCEWithLogitsLoss() if task_type == "multilabel" else nn.L1Loss()
        ckpt_path  = f"best_model_{model_class.__name__}_seed{seed}.pt"

        param_count = count_params(model)
        print(f"Parameters: {param_count}")
        print(f"Checkpoint: {ckpt_path}")

        best_val          = 0.0 if task_type == "multilabel" else float("inf")
        best_epoch        = 0
        epochs_no_improve = 0
        gate_checked      = False
        gate_collapsed    = False

        torch.cuda.reset_peak_memory_stats()
        start_time = time.time()

        for epoch in range(1, max_epochs + 1):

            # ---- Gate check at epoch 6 ----
            if epoch == 6 and not gate_checked:
                gate_collapsed = check_gate_health(model, val_loader, device)
                gate_checked   = True
                if gate_collapsed:
                    print("  Resetting optimizer to lr=5e-4.")
                    optimizer = optim.AdamW(model.parameters(), lr=5e-4, weight_decay=1e-4)
                    scheduler = optim.lr_scheduler.CosineAnnealingLR(
                        optimizer, T_max=max_epochs - epoch, eta_min=1e-5
                    )

            # ---- Training ----
            model.train()
            total_loss  = 0
            epoch_start = time.time()
            optimizer.zero_grad()

            pbar = tqdm(
                enumerate(train_loader),
                total=len(train_loader),
                desc=f"Seed {seed} | Epoch {epoch}",
                leave=False
            )

            for step, batch in pbar:
                batch = batch.to(device)
                out   = model(batch)

                if task_type == "multilabel":
                    loss = criterion(out, batch.y.float())
                else:
                    loss = criterion(out.squeeze(), batch.y.squeeze())

                loss = loss / accum_steps
                loss.backward()

                total_loss += loss.item() * accum_steps

                if (step + 1) % accum_steps == 0 or (step + 1) == len(train_loader):
                    torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
                    optimizer.step()
                    optimizer.zero_grad()

                pbar.set_postfix({"Loss": f"{loss.item() * accum_steps:.4f}"})

            scheduler.step()
            avg_loss   = total_loss / len(train_loader)
            current_lr = scheduler.get_last_lr()[0]

            # ---- Val only during training ----
            if task_type == "multilabel":
                val_metric = evaluate_multilabel(model, val_loader)
                improved   = val_metric > best_val
            else:
                val_metric = evaluate_regression(model, val_loader)
                improved   = val_metric < best_val

            epoch_time = time.time() - epoch_start

            print(
                f"Epoch {epoch:03d} | "
                f"Loss {avg_loss:.4f} | "
                f"Val {val_metric:.4f} | "
                f"LR {current_lr:.6f} | "
                f"Time {epoch_time:.2f}s"
            )

            # ---- Checkpoint + early stopping ----
            if improved:
                best_val          = val_metric
                best_epoch        = epoch
                epochs_no_improve = 0
                torch.save(model.state_dict(), ckpt_path)
            else:
                epochs_no_improve += 1

            if epochs_no_improve >= patience:
                print(f"Early stopping at epoch {epoch}.")
                break

        # ---- Single clean test evaluation ----
        print(f"\nLoading best checkpoint (epoch {best_epoch})...")
        model.load_state_dict(torch.load(ckpt_path))

        if task_type == "multilabel":
            test_ap = evaluate_multilabel(model, test_loader)
        else:
            test_ap = evaluate_regression(model, test_loader)

        total_time = time.time() - start_time
        peak_mem   = torch.cuda.max_memory_allocated() / 1024**2

        print("-" * 60)
        print(f"Seed {seed} Results:")
        print(f"  Best Val:     {best_val:.4f}")
        print(f"  Test AP:      {test_ap:.4f}")   # ← the only number that matters
        print(f"  Best Epoch:   {best_epoch}")
        print(f"  Total Time:   {total_time:.2f}s")
        print(f"  Peak Memory:  {peak_mem:.2f} MB")
        print("-" * 60)

        results.append(test_ap)

        log_results_to_csv(
            "results.csv",
            {
                "Dataset"        : dataset_name,
                "Model"          : model_class.__name__,
                "HiddenDim"      : model_kwargs.get("hidden_dim"),
                "NumLayers"      : model_kwargs.get("num_layers"),
                "Params"         : param_count,
                "Seed"           : seed,
                "BestVal"        : best_val,
                "TestAP"         : test_ap,
                "BestEpoch"      : best_epoch,
                "TotalTime"      : total_time,
                "PeakMemoryMB"   : peak_mem,
                "GateCollapsed"  : gate_collapsed,
                "MaxEpochs"      : max_epochs,
                "Patience"       : patience,
                "BatchSize"      : batch_size,
                "AccumSteps"     : accum_steps,
                "EffectiveBatch" : batch_size * accum_steps,
            }
        )

    mean = np.mean(results)
    std  = np.std(results)

    print("=" * 60)
    print(f"FINAL Results across {len(seeds)} seeds:")
    print(f"  Test AP: {mean:.4f} ± {std:.4f}")
    if std > 0.020:
        print("  ⚠️  High variance — check eigenvector stability.")
    elif std < 0.012:
        print("  ✅ Low variance — sign fix working correctly.")
    else:
        print("  ⚠️  Moderate variance — acceptable.")
    print("=" * 60)

    return mean, std

In [ ]:
mean, std = train_model(
    model_class  = HybridGraphFNet_Best_Peptides,
    model_kwargs = {"hidden_dim": 128, "num_layers": 4, "out_dim": 10},
    dataset_name = "LRGB",
    max_epochs   = 120,   # safe for Kaggle, model converges by ~80
    patience     = 20,
    batch_size   = 8,
    accum_steps  = 4,
    seeds        = [0, 1, 2],
)

Dataset: LRGB | Seed: 0
Batch size: 8 | Accum steps: 4 | Effective batch: 32
Parameters: 328603


Epoch 001 | Loss 0.3628 | Val 0.2901 | Test 0.2880 | LR 0.001000 | Time 87.25s


Epoch 002 | Loss 0.3258 | Val 0.3387 | Test 0.3364 | LR 0.001000 | Time 85.66s


Epoch 003 | Loss 0.2942 | Val 0.3975 | Test 0.3875 | LR 0.000999 | Time 84.97s


Epoch 004 | Loss 0.2755 | Val 0.4211 | Test 0.4062 | LR 0.000999 | Time 85.31s


...

Epoch 067 | Loss 0.0346 | Val 0.6336 | Test 0.6302 | LR 0.000750 | Time 85.35s


Epoch 068 | Loss 0.0337 | Val 0.6367 | Test 0.6166 | LR 0.000743 | Time 85.38s


Epoch 069 | Loss 0.0340 | Val 0.6267 | Test 0.6240 | LR 0.000737 | Time 85.03s


Epoch 070 | Loss 0.0330 | Val 0.6353 | Test 0.6306 | LR 0.000730 | Time 85.24s


Seed 0 | Epoch 71:  34%|███▍      | 459/1360 [00:22<00:42, 21.13it/s, Loss=0.0640]